# Structured State Space Models (S4-style)

In this lab, you will explore continuous-time structured state space models (SSMs):
- Implement a continuous-time diagonal SSM and discretize it
- Train it in RNN-style by unrolling through time
- Reinterpret the same model as a global convolution
- Accelerate the convolution using FFT
- Compare runtime performance between approaches

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"

## Part 1: Toy Forecasting Dataset

**Question 1.** Understand the `ToyForecastDataset` class which generates
simple sinusoidal forecasting data. We will use this to train and evaluate
our different SSM implementations.

In [ ]:
class ToyForecastDataset(Dataset):
    """
    Simple sinusoidal forecasting dataset.
    Predict next timestep.
    """
    def __init__(self, n_samples=1000, seq_len=128):
        self.seq_len = seq_len
        self.data = []
        self.dt = 10. / seq_len

        for _ in range(n_samples):
            t = np.linspace(0, 10, seq_len + 1)
            signal = (
                np.sin(2 * t)
                + 0.5 * np.sin(5 * t)
                + 0.1 * np.random.randn(len(t))
            )
            self.data.append(signal.astype(np.float32))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data = self.data[idx]
        return (
            torch.tensor(data[:-1]).unsqueeze(-1),
            torch.tensor(data[1:]).unsqueeze(-1),
            self.dt
        )

# Create a dataset and dataloader
dataset = ToyForecastDataset(n_samples=100, seq_len=1024)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

## Part 2: Continuous-time SSM (Diagonal A)

**Question 2.** Implement the discretization method for a continuous-time diagonal SSM.
The continuous-time system is:
```
    h'(t) = A h(t) + B x(t)
    o(t)  = C h(t)
```

with diagonal A. We discretize using:
```
    A_d = exp(dt A)
    B_d = (exp(dt A) - 1) / A * B
```

Then train it in RNN-style by unrolling:
```
    h[t+1] = A_d * h[t] + B_d * x[t]
    o[t]   = C * h[t]
```

In [ ]:
class DiagonalSSM(nn.Module):
    def __init__(self, d_in=1, d_hidden=32):
        super().__init__()
        self.d_hidden = d_hidden
        # Init. A, B, and C
        self.A = nn.Parameter(torch.randn(d_hidden))
        self.B = nn.Parameter(torch.randn(d_hidden, d_in) * 0.1)
        self.C = nn.Parameter(torch.randn(d_in, d_hidden) * 0.1)

    def A_d(self, dt):
        # TODO: Compute discrete A_d
        raise NotImplementedError()

    def B_d(self, dt):
        # TODO: Compute discrete B_d
        raise NotImplementedError()

    def forward(self, x, dt=0.1):
        """
        x: (B, L, d_model)
        dt: integration step size
        return: (B, L, d_model)
        """
        return self._forward_rnn(x, dt)

    def _forward_rnn(self, x, dt):
        """RNN-style forward: h[t+1] = A_d * h[t] + B_d * x[t]"""
        B, L, _ = x.shape
        A_d, B_d = self.A_d(dt), self.B_d(dt)
        h = torch.zeros(B, self.d_hidden, device=x.device)
        os = []
        for t in range(L):
            x_t = x[:, t]
            h = A_d * h + torch.matmul(x_t, B_d.transpose(0, 1))
            o = torch.matmul(h, self.C.t())
            os.append(o)
        return torch.stack(os, dim=1)

**Question 3.** Now train your model on the following dataset.
What do you observe?

In [ ]:
def train(model, loader, epochs=10, lr=1e-3, forward_options=None):
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0

        for x, o, dt in loader:
            x, o, dt = x.to(device), o.to(device), dt.to(device)
            dt = dt[0].to(x.dtype)

            if forward_options is None:
                pred = model(x, dt)
            else:
                pred = model(x, dt, **forward_options)
            loss = ((pred - o) ** 2).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()

        print(f"Epoch {epoch:02d} | Loss: {total_loss/len(loader):.4f}")

# Initialize and train the RNN model
model = DiagonalSSM(d_in=1, d_hidden=32)
train(model, loader)

**Question 4.** What do you think is wrong with the current
parametrization of your model? Fix that and check that it
solves the explosion of h[t] along time.

In [ ]:
class DiagonalSSM(nn.Module):
    """
    Continuous-time SSM:
        h'(t) = A h(t) + B x(t)
        o(t)  = C h(t)

    with diagonal A.

    Trained in RNN-style after discretization.
    """

    def __init__(self, d_in=1, d_hidden=32):
        super().__init__()

        self.d_hidden = d_hidden
        # Stable init
        # TODO here: define alpha
        # self.alpha = ...
        self.B = nn.Parameter(torch.randn(d_hidden, d_in) * 0.1)
        self.C = nn.Parameter(torch.randn(d_in, d_hidden) * 0.1)

    @property
    def A(self):
        # TODO here: compute A based on alpha such that A is 
        # constrained to be a vector of negative values
        raise NotImplementedError("Implement A property.")

    def A_d(self, dt):
        """
        Compute discrete A_d.
        A_d = exp(dt A)
        
        TODO: Implement discretization.
        """
        raise NotImplementedError("Implement A_d method.")

    def B_d(self, dt):
        """
        Compute discrete B_d.
        B_d = (exp(dt A) - 1)/A * B
        
        TODO: Implement discretization.
        """
        raise NotImplementedError("Implement B_d method.")

    def forward(self, x, dt=0.1):
        """
        x: (B, L, d_model)
        dt: integration step size
        return: (B, L, d_model)
        """
        return self._forward_rnn(x, dt)

    def _forward_rnn(self, x, dt):
        """RNN-style forward: h[t+1] = A_d * h[t] + B_d * x[t]"""
        B, L, _ = x.shape
        A_d, B_d = self.A_d(dt), self.B_d(dt)
        h = torch.zeros(B, self.d_hidden, device=x.device)
        os = []
        for t in range(L):
            x_t = x[:, t]
            h = A_d * h + torch.matmul(x_t, B_d.transpose(0, 1))
            o = torch.matmul(h, self.C.t())
            os.append(o)
        return torch.stack(os, dim=1)

## Part 3: Convolution View

**Question 5.** The same SSM can be viewed as a global convolution with kernel:
```
    K_k = C A_d^k B_d
```

Extend the `DiagonalSSM` class to support a conv-style forward pass
that uses the kernel directly instead of unrolling through time.

## Part 4: FFT Convolution

**Question 6.** Convolution in time domain becomes element-wise multiplication in
frequency domain. Implement FFT-based convolution to accelerate computations:
```
    O = IFFT(FFT(X) ⊙ FFT(K))
```

where ⊙ denotes element-wise multiplication.

In [ ]:
def fft_convolve(x, K):
    """
    x: (B, L, d_in)
    K: (L, d_in, d_out)
    """

    B, L, d = x.shape
    fft_size = 2 * L

    x_f = torch.fft.rfft(x, n=fft_size, dim=1)
    K_f = torch.fft.rfft(K, n=fft_size, dim=0)

    o_f = torch.einsum("bli, lio -> blo", x_f, K_f)
    o = torch.fft.irfft(o_f, n=fft_size, dim=1)

    return o[:, :L]

## Part 5: Training and Evaluation

**Question 7.** Train the RNN-style and Conv-style versions on the toy dataset.
Compare their training curves.

**Question 8.** Compare the runtime performance of RNN-style vs Conv-style SSM at inference.
Which one is faster? Can you explain why?

In [ ]:
x, _, dt = next(iter(loader))
x = x.to(device)
dt = dt.to(device)[0].to(x.dtype)
n_repeats = 1000

print("\nTiming comparison (forward pass)...")

# TODO here

print("\nDone.")